# Task 1 EDA: News Acquisition & Filtering

Run after the full pipeline (`gdelt.fetch` → `selection.build_queue` → `scrapers.run_all` → `preprocessing.*` → `sentiment.*`) has produced real data. This notebook is the evidence behind the filtering/design choices documented in the README's "Design decisions" section -- it's meant to be looked at and reasoned about, not just executed once.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))
from common.config import load_config

config = load_config()
processed_dir = Path(config["paths"]["processed_dir"])
raw_dir = Path(config["paths"]["raw_dir"])

## 1. Coverage: headlines per day per source, across all 5 years
Gaps here mean either GDELT has no coverage for that period, or the theme/section filter dropped everything that day -- worth knowing which before trusting the daily sentiment series.

In [ ]:
id_headlines = pd.read_parquet(processed_dir / "id_headlines_filtered.parquet")
id_headlines["wib_date"] = pd.to_datetime(id_headlines["wib_date"])

daily_counts = id_headlines.groupby([id_headlines["wib_date"].dt.to_period("M"), "site"]).size().unstack(fill_value=0)
daily_counts.plot(figsize=(14, 4), title="Filtered ID headlines per month per source")
plt.ylabel("headline count")
plt.show()

## 2. Filter funnel
How much each stage (URL dedupe, section filter, theme filter, daily cap, body-length filter) drops -- this is the direct evidence for the "strategic preprocessing" writeup.

In [ ]:
with open(processed_dir / "filter_report.json") as f:
    report = json.load(f)

funnel = pd.DataFrame(report["stages"])
display(funnel)

plt.figure(figsize=(10, 4))
plt.barh(funnel["stage"], funnel["count"])
plt.title("Filter funnel")
plt.gca().invert_yaxis()
plt.show()

## 3. Top GDELT themes: kept vs. dropped
Sanity-checks whether `relevant_theme_prefixes` in `config.yaml` is actually capturing the right articles, or needs tuning.

In [ ]:
def top_themes(df, n=20):
    tokens = df["V2Themes"].dropna().str.split(";").explode().str.split(",").str[0]
    return tokens.value_counts().head(n)

print("Top themes among KEPT headlines:")
display(top_themes(id_headlines))

## 4. Scrape success rate per site
Compares the scrape queue size against how many articles actually got a body (vs. failed fetch, missing title/body, or dropped for being too short).

In [ ]:
queue = pd.read_parquet(Path(config["paths"]["interim_dir"]) / "scrape_queue.parquet")
id_articles = pd.read_parquet(processed_dir / "id_articles.parquet")

success = id_articles.groupby("source")["body"].apply(lambda s: s.notna().mean())
queued = queue.groupby("site").size()
pd.DataFrame({"queued": queued, "body_success_rate": success}).fillna(0)

## 5. Sentiment distributions

In [ ]:
id_scored = pd.read_parquet(processed_dir / "id_articles_scored.parquet")
global_scored = pd.read_parquet(processed_dir / "global_headlines_scored.parquet")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
id_scored["title_sentiment"].hist(bins=30, ax=axes[0])
axes[0].set_title("ID title sentiment")
global_scored["title_sentiment"].hist(bins=30, ax=axes[1])
axes[1].set_title("Global title sentiment")
plt.show()

## 6. Title vs. body sentiment agreement
For articles with both a title and a scraped body: how often do they agree on sign? A low agreement rate is a finding worth writing up -- it would argue headline-only sentiment (the GDELT-only alternative considered and rejected) loses real signal.

In [ ]:
has_body = id_scored["body_sentiment"].notna()
sub = id_scored[has_body].copy()
sub["body_sentiment"] = sub["body_sentiment"].astype(float)
agree = (sub["title_sentiment"] > 0) == (sub["body_sentiment"] > 0)
print(f"n={len(sub)}, sign agreement={agree.mean():.1%}")

plt.figure(figsize=(5, 5))
plt.scatter(sub["title_sentiment"], sub["body_sentiment"], alpha=0.3, s=10)
plt.xlabel("title sentiment")
plt.ylabel("body sentiment")
plt.axhline(0, color="grey", lw=0.5)
plt.axvline(0, color="grey", lw=0.5)
plt.title("Title vs body sentiment")
plt.show()

## 7. Daily sentiment series: Indonesian vs. global

In [ ]:
daily = pd.read_csv(processed_dir / "daily_news_features.csv", parse_dates=["date"])

fig, ax = plt.subplots(figsize=(14, 4))
daily.set_index("date")[["id_title_sent_mean", "global_title_sent_mean"]].rolling(14).mean().plot(ax=ax)
ax.set_title("14-day rolling mean sentiment: Indonesian vs. global headlines")
ax.axhline(0, color="grey", lw=0.5)
plt.show()

## 8. Sample kept vs. dropped headlines
For manually judging whether the section + theme filter is doing the right thing -- pull a fresh random sample each time you review this.

In [ ]:
raw = pd.concat([pd.read_parquet(p) for p in raw_dir.glob("gdelt_id_*.parquet")], ignore_index=True)
kept_urls = set(id_headlines["url"])

print("KEPT sample:")
display(id_headlines.sample(min(20, len(id_headlines)))[["title", "site", "section"]])

print("DROPPED sample:")
dropped = raw[~raw["url"].isin(kept_urls)]
display(dropped.sample(min(20, len(dropped)))[["title", "SourceCommonName"]])